# Create Flag Param

In [0]:
dbutils.widgets.text('incremental_flag','0')
incremental_flag = dbutils.widgets.get('incremental_flag')

### Creating Dimension Model

In [0]:
df_car_sales = (
    spark.read.format('parquet')
    .option('header','true')
    .load('abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/silver/car_sales')
)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_car_sales = (
    df_car_sales
    .dropDuplicates(['model_id'])
    .select("Model_ID","Model_Category")
)

### Dim_model sink Initial and Incremental

In [0]:
df_silver = spark.read.parquet("abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/silver/car_sales")

df_silver.createOrReplaceTempView("temp_silver_car_sales")

if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    df_sink = spark.sql('''
            SELECT dim_model_key, Model_ID, Model_Category
            FROM cars_catalog.gold.dim_model
            ''')
else:
    df_sink = spark.sql('''
            SELECT 1 AS dim_model_key, Model_ID, Model_Category
            FROM temp_silver_car_sales
            WHERE 1=2
            ''')

### Filtering new records and old records

In [0]:
df_filter = (
    df_car_sales
    .join(df_sink, df_car_sales.Model_ID == df_sink.Model_ID, 'left')
    .select(df_car_sales.Model_ID, df_car_sales.Model_Category, df_sink.dim_model_key)
    )

Filtering out old records from join result

In [0]:
df_filter_old = df_filter.filter(col("dim_model_key").isNotNull())

Filtering out new records from join result

In [0]:
df_filter_new = df_filter.filter(col("dim_model_key").isNull()).select(df_car_sales.Model_ID, df_car_sales.Model_Category)

Create Surrogate key

In [0]:
if(incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("select max(dim_model_key) from cars_catalog.gold.dim_model")
    max_value = max_value_df.collect()[0][0]+1

In [0]:
df_filter_new = (
    df_filter_new
    .withColumn("dim_model_key", max_value + monotonically_increasing_id())
)

### Create Final df = old + new

In [0]:
df_final = df_filter_new.union(df_filter_old)

# SCD Type-1 (UPSERT)

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    # Incremental run
    delta_tbl = DeltaTable.forPath(spark, 'abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/gold/dim_model')
    
    (
    delta_tbl.alias('trg').merge(
        df_final.alias('src'),
        "trg.dim_model_key = src.dim_model_key"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
    )

else:
    # Initial run case
    (
        df_final.write
        .format('delta')
        .mode('append')
        .option("path","abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/gold/dim_model")
        .saveAsTable('cars_catalog.gold.dim_model')
    )